In [1]:
import torch
import numpy as np
from datasets import load_dataset
from transformers import pipeline
from sklearn.metrics import accuracy_score, f1_score, classification_report
from tqdm.auto import tqdm

torch_device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
pipeline_device = torch_device if torch_device.type == "mps" else -1
print("device:", torch_device)


device: mps


In [2]:
model_name = "typeform/distilbert-base-uncased-mnli"

classifier = pipeline(
    "zero-shot-classification",
    model=model_name,
    device=pipeline_device,
)

print(model_name)
print(classifier.model.config.id2label)


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

typeform/distilbert-base-uncased-mnli
{0: 'ENTAILMENT', 1: 'NEUTRAL', 2: 'CONTRADICTION'}


In [3]:
ds = load_dataset("glue", "mrpc", split="validation")
print(ds)
print(ds[0])


Dataset({
    features: ['sentence1', 'sentence2', 'label', 'idx'],
    num_rows: 408
})
{'sentence1': "He said the foodservice pie business doesn 't fit the company 's long-term growth strategy .", 'sentence2': '" The foodservice pie business does not fit our long-term growth strategy .', 'label': 1, 'idx': 9}


In [4]:
sent1 = ds["sentence1"]
sent2 = ds["sentence2"]
y_true = np.array(ds["label"])
sequences = [f"Sentence 1: {s1}\nSentence 2: {s2}" for s1, s2 in zip(sent1, sent2)]

print("num_examples:", len(y_true))
print("positive_rate:", y_true.mean())
print(sequences[0])


num_examples: 408
positive_rate: 0.6838235294117647
Sentence 1: He said the foodservice pie business doesn 't fit the company 's long-term growth strategy .
Sentence 2: " The foodservice pie business does not fit our long-term growth strategy .


In [5]:
candidate_labels = ["paraphrase", "not paraphrase"]
hypothesis_template = "These two sentences are {}."
label_to_id = {"not paraphrase": 0, "paraphrase": 1}

batch_size = 16
all_outputs = []

for i in tqdm(range(0, len(sequences), batch_size)):
    batch_sequences = sequences[i:i + batch_size]
    batch_outputs = classifier(
        batch_sequences,
        candidate_labels=candidate_labels,
        hypothesis_template=hypothesis_template,
        multi_label=False,
        batch_size=batch_size,
        truncation=True,
    )
    all_outputs.extend(batch_outputs)

y_pred = np.array([label_to_id[out["labels"][0]] for out in all_outputs])
print("done")


  0%|          | 0/26 [00:00<?, ?it/s]

done


In [6]:
acc = accuracy_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)

print({"accuracy": acc, "f1": f1})
print(classification_report(y_true, y_pred, target_names=["not_paraphrase", "paraphrase"]))


{'accuracy': 0.6299019607843137, 'f1': 0.7470686767169179}
                precision    recall  f1-score   support

not_paraphrase       0.38      0.26      0.31       129
    paraphrase       0.70      0.80      0.75       279

      accuracy                           0.63       408
     macro avg       0.54      0.53      0.53       408
  weighted avg       0.60      0.63      0.61       408



In [7]:
for i in range(5):
    out = all_outputs[i]
    score_map = {label: float(score) for label, score in zip(out["labels"], out["scores"])}
    print("=" * 80)
    print("sentence1:", sent1[i])
    print("sentence2:", sent2[i])
    print("true:", int(y_true[i]), "pred:", int(y_pred[i]), "top_label:", out["labels"][0])
    print("scores:", score_map)


sentence1: He said the foodservice pie business doesn 't fit the company 's long-term growth strategy .
sentence2: " The foodservice pie business does not fit our long-term growth strategy .
true: 1 pred: 1 top_label: paraphrase
scores: {'paraphrase': 0.533607542514801, 'not paraphrase': 0.46639248728752136}
sentence1: Magnarelli said Racicot hated the Iraqi regime and looked forward to using his long years of training in the war .
sentence2: His wife said he was " 100 percent behind George Bush " and looked forward to using his years of training in the war .
true: 0 pred: 1 top_label: paraphrase
scores: {'paraphrase': 0.5474331974983215, 'not paraphrase': 0.45256683230400085}
sentence1: The dollar was at 116.92 yen against the yen , flat on the session , and at 1.2891 against the Swiss franc , also flat .
sentence2: The dollar was at 116.78 yen JPY = , virtually flat on the session , and at 1.2871 against the Swiss franc CHF = , down 0.1 percent .
true: 0 pred: 1 top_label: paraphrase

In [8]:
mistakes = np.where(y_true != y_pred)[0][:10]
print("num_errors:", int((y_true != y_pred).sum()))

for i in mistakes:
    out = all_outputs[int(i)]
    score_map = {label: float(score) for label, score in zip(out["labels"], out["scores"])}
    print("=" * 80)
    print("idx:", int(i))
    print("sentence1:", sent1[i])
    print("sentence2:", sent2[i])
    print("true:", int(y_true[i]), "pred:", int(y_pred[i]), "top_label:", out["labels"][0])
    print("scores:", score_map)


num_errors: 151
idx: 1
sentence1: Magnarelli said Racicot hated the Iraqi regime and looked forward to using his long years of training in the war .
sentence2: His wife said he was " 100 percent behind George Bush " and looked forward to using his years of training in the war .
true: 0 pred: 1 top_label: paraphrase
scores: {'paraphrase': 0.5474331974983215, 'not paraphrase': 0.45256683230400085}
idx: 2
sentence1: The dollar was at 116.92 yen against the yen , flat on the session , and at 1.2891 against the Swiss franc , also flat .
sentence2: The dollar was at 116.78 yen JPY = , virtually flat on the session , and at 1.2871 against the Swiss franc CHF = , down 0.1 percent .
true: 0 pred: 1 top_label: paraphrase
scores: {'paraphrase': 0.5974612236022949, 'not paraphrase': 0.4025387167930603}
idx: 3
sentence1: The AFL-CIO is waiting until October to decide if it will endorse a candidate .
sentence2: The AFL-CIO announced Wednesday that it will decide in October whether to endorse a candi

In [9]:
summary = {
    "dataset": "glue/mrpc",
    "split": "validation",
    "model": model_name,
    "device": str(torch_device),
    "num_examples": len(ds),
    "accuracy": float(acc),
    "f1": float(f1),
}
summary


{'dataset': 'glue/mrpc',
 'split': 'validation',
 'model': 'typeform/distilbert-base-uncased-mnli',
 'device': 'mps',
 'num_examples': 408,
 'accuracy': 0.6299019607843137,
 'f1': 0.7470686767169179}